In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from transformers import AutoTokenizer

In [ ]:
INPUT_PATH = "../data/processed/QA_data_evaluated.parquet"
TOKENIZER_MODEL = "vinai/bartpho-syllable"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
df = pd.read_parquet(INPUT_PATH, engine="pyarrow")

# Normalize: convert numpy arrays in nested structure to Python lists
# (parquet stores chunks and qa_pairs as numpy.ndarray, code expects lists)
def _normalize_chunks(chunks):
    if isinstance(chunks, np.ndarray):
        chunks = list(chunks)
    for chunk in chunks:
        if isinstance(chunk, dict) and 'qa_pairs' in chunk:
            qa = chunk['qa_pairs']
            if isinstance(qa, np.ndarray):
                chunk['qa_pairs'] = list(qa)
    return chunks

df['chunks'] = df['chunks'].apply(_normalize_chunks)

In [ ]:
# Check answers that are "Thông tin không có trong văn bản" (before punctuation filter)
NO_INFO_PATTERNS = ["Thông tin không có trong văn bản"]

no_info_count = 0
no_info_assessments = {}

for _, row in df.iterrows():
    for chunk in row.get('chunks', []):
        if not isinstance(chunk, dict):
            continue
        for qa in chunk.get('qa_pairs', []):
            ans = qa.get('answer', '').strip().rstrip('.')
            if ans in NO_INFO_PATTERNS:
                no_info_count += 1
                a = qa.get('overall_assessment', 'UNKNOWN')
                no_info_assessments[a] = no_info_assessments.get(a, 0) + 1

total_qa = sum(
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
)

print("=" * 60)
print(f'Answers matching: "{NO_INFO_PATTERNS[0]}"')
print("=" * 60)
if total_qa > 0:
    print(f"Count: {no_info_count}/{total_qa} ({no_info_count/total_qa:.2%})")
else:
    print("No QA pairs found in dataset.")
print(f"\nAssessment breakdown:")
for a, c in sorted(no_info_assessments.items(), key=lambda x: -x[1]):
    pct = f" ({c/no_info_count:.1%})" if no_info_count > 0 else ""
    print(f"  {a}: {c}{pct}")
print("=" * 60)

In [ ]:
# Filter: context/answer must end with '.', question must end with '?'
# - Bad context  → remove entire chunk (all QA pairs in that chunk)
# - Bad question/answer → remove only that QA pair, keep the chunk if others remain

before_articles = len(df)
before_chunks = sum(len(row.get('chunks', [])) for _, row in df.iterrows())
before_qa = sum(
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
)

bad_context_chunks, bad_context_qa = 0, 0
bad_context_assessments, bad_context_samples = {}, []
bad_question_count, bad_answer_count = 0, 0
bad_question_assessments, bad_answer_assessments = {}, {}
bad_question_samples, bad_answer_samples = [], []

for idx, row in df.iterrows():
    good_chunks = []
    for chunk in row.get('chunks', []):
        if not isinstance(chunk, dict):
            continue
        context = chunk.get('context', '')

        # === 1. Context check — remove entire chunk ===
        if not context.strip().endswith('.'):
            bad_context_chunks += 1
            qa_in_chunk = chunk.get('qa_pairs', [])
            bad_context_qa += len(qa_in_chunk)
            if len(bad_context_samples) < 3:
                bad_context_samples.append(context.strip()[-100:])
            for qa in qa_in_chunk:
                a = qa.get('overall_assessment', 'UNKNOWN')
                bad_context_assessments[a] = bad_context_assessments.get(a, 0) + 1
            continue  # skip this chunk

        # === 2. Question/answer check — remove individual QA pairs ===
        good_pairs = []
        for qa in chunk.get('qa_pairs', []):
            if not isinstance(qa, dict):
                continue
            q_ok = qa.get('question', '').strip().endswith('?')
            a_ok = qa.get('answer', '').strip().endswith('.')
            assess = qa.get('overall_assessment', 'UNKNOWN')
            if q_ok and a_ok:
                good_pairs.append(qa)
            else:
                if not q_ok:
                    bad_question_count += 1
                    bad_question_assessments[assess] = bad_question_assessments.get(assess, 0) + 1
                    if len(bad_question_samples) < 3:
                        bad_question_samples.append(qa.get('question', ''))
                if not a_ok:
                    bad_answer_count += 1
                    bad_answer_assessments[assess] = bad_answer_assessments.get(assess, 0) + 1
                    if len(bad_answer_samples) < 3:
                        bad_answer_samples.append(qa.get('answer', ''))

        if good_pairs:
            chunk['qa_pairs'] = good_pairs
            good_chunks.append(chunk)

    df.at[idx, 'chunks'] = good_chunks

# Remove articles with no chunks left
df = df[df['chunks'].map(len) > 0].reset_index(drop=True)

after_articles = len(df)
after_qa = sum(
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
)

# === Report ===
print("=" * 70)
print("PUNCTUATION FILTER REPORT")
print("=" * 70)

print(f"\n[Context not ending with '.']  {bad_context_chunks} chunks → {bad_context_qa} QA pairs removed")
print(f"  Assessment: {bad_context_assessments}")
for i, s in enumerate(bad_context_samples, 1):
    print(f"  Sample {i}: ...{s}")

print(f"\n[Question not ending with '?'] {bad_question_count} QA pairs removed")
print(f"  Assessment: {bad_question_assessments}")
for i, s in enumerate(bad_question_samples, 1):
    print(f"  Sample {i}: {s.strip()[:120]}")

print(f"\n[Answer not ending with '.']   {bad_answer_count} QA pairs removed")
print(f"  Assessment: {bad_answer_assessments}")
for i, s in enumerate(bad_answer_samples, 1):
    print(f"  Sample {i}: {s.strip()[:120]}")

all_assessments = {}
for d in [bad_context_assessments, bad_question_assessments, bad_answer_assessments]:
    for k, v in d.items():
        all_assessments[k] = all_assessments.get(k, 0) + v

print(f"\n--- Total Removed Assessment Breakdown ---")
for a, c in sorted(all_assessments.items(), key=lambda x: -x[1]):
    print(f"  {a}: {c}")

print(f"\n--- Overall Summary ---")
print(f"Articles: {after_articles:,}/{before_articles:,} ({before_articles - after_articles:,} removed)")
if before_qa > 0:
    print(f"QA pairs: {after_qa:,}/{before_qa:,} ({before_qa - after_qa:,} removed, {(before_qa - after_qa)/before_qa:.2%})")
print("=" * 70)

# 1. Basic statistics

In [ ]:
def display_schema(df):
    sample = df.iloc[0]
    for col in df.columns:
        col_dtype = type(sample[col])
        print(f">- Name: {col}\t- Dtype: {col_dtype}")
        if col == 'chunks' and isinstance(sample[col], list) and len(sample[col]) > 0:
            chunk = sample[col][0]
            print(f"  >- chunks[0] keys: {list(chunk.keys())}")
            if 'qa_pairs' in chunk and chunk['qa_pairs']:
                qa_pair = chunk['qa_pairs'][0]
                main_attributes = ("question", "answer", "overall_assessment")
                for key, value in qa_pair.items():
                    label = "(main)" if key in main_attributes else ""
                    print(f"    >- Name: {key}\tDtype: {type(value)} {label}")

In [ ]:
# Extract topic_id from question prefix (e.g. "FACTOID-What is...")
# Filter out QA pairs whose question lacks a recognized topic prefix.

def extract_topic_id(question: str):
    pattern = re.compile(r'^(FACTOID|SUMMARY|COMPARISON|VERIFICATION)-(.+)')
    match = pattern.match(question)
    return match.group(1) if match else None

num_rows = len(df)
num_qa_pairs = sum(
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
)

# Stamp topic_id on every QA pair, then remove those without one
for idx, row in df.iterrows():
    updated_chunks = []
    for chunk in row.get('chunks', []):
        if not isinstance(chunk, dict):
            continue
        for qa in chunk.get('qa_pairs', []):
            qa['topic_id'] = extract_topic_id(qa.get('question', ''))
        chunk['qa_pairs'] = [qa for qa in chunk.get('qa_pairs', []) if qa.get('topic_id') is not None]
        if chunk['qa_pairs']:
            updated_chunks.append(chunk)
    df.at[idx, 'chunks'] = updated_chunks

df = df[df['chunks'].map(len) > 0].reset_index(drop=True)

_num_qa_pairs_after = sum(
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
)
_qa_per_chunk = [
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
]

removed_rows_count = num_rows - len(df)
removed_qa_pairs_count = num_qa_pairs - _num_qa_pairs_after

print("===== After topic_id filtering =====")
print(f"Total articles: {len(df):,}/{num_rows:,}, {removed_rows_count:,} ({removed_rows_count/num_rows:.2%}) removed.")
print(f"Total QA pairs: {_num_qa_pairs_after:,}/{num_qa_pairs:,}, {removed_qa_pairs_count:,} ({removed_qa_pairs_count/num_qa_pairs:.2%}) removed")
print("=" * 40)
print("QA pairs per chunk analysis:")
print(f"\tMean:   {np.mean(_qa_per_chunk):.2f}")
print(f"\tMedian: {np.median(_qa_per_chunk):.1f}")
print(f"\tStd Dev:{np.std(_qa_per_chunk):.2f}")
print(f"\tMin:    {np.min(_qa_per_chunk)}")
print(f"\tMax:    {np.max(_qa_per_chunk)}")
print(f"\t25%:    {np.percentile(_qa_per_chunk, 25):.0f}")
print(f"\t50%:    {np.percentile(_qa_per_chunk, 50):.0f}")
print(f"\t75%:    {np.percentile(_qa_per_chunk, 75):.0f}")
print(f"\t90%:    {np.percentile(_qa_per_chunk, 90):.0f}")
print("=" * 40)
print("Schema:")
display_schema(df)

In [ ]:
# Flatten nested structure (articles → chunks → qa_pairs) into a per-QA-pair DataFrame
# Optionally deduplicates by context text to avoid identical chunks appearing across articles.

dedupe_context = True

meta_cols = ['url', 'title', 'time']
qa_fields = [
    'question', 'answer', 'topic_id',
    'overall_assessment', 'overall_reason', 'total_score',
    'answerability_score', 'answerability_reason',
    'answer_accuracy_score', 'answer_accuracy_reason',
    'clarity_score', 'clarity_reason',
    'conciseness_score', 'conciseness_reason',
    'usefulness_score', 'usefulness_reason',
]

rows = []
seen_contexts = set()

for _, row in df.iterrows():
    meta = {col: row.get(col) for col in meta_cols}
    for chunk in row.get('chunks', []):
        if not isinstance(chunk, dict):
            continue
        context = chunk.get('context', '')
        if not context:
            continue
        if dedupe_context:
            if context in seen_contexts:
                continue
            seen_contexts.add(context)
        for qa in chunk.get('qa_pairs', []):
            if not isinstance(qa, dict):
                continue
            entry = {**meta, 'context': context}
            for field in qa_fields:
                entry[field] = qa.get(field)
            rows.append(entry)

df_flat = pd.DataFrame(rows)
total_evaluated = len(df_flat)
assessment_labels = df_flat['overall_assessment'].dropna().unique().tolist()
print(f"[INFO] Flattened to {total_evaluated:,} QA pairs from {df_flat['url'].nunique():,} unique articles")
print(f"[INFO] Unique contexts: {df_flat['context'].nunique():,}")
print(f"[INFO] Assessment labels found: {assessment_labels}")

# 2. Token Length Analysis

In [ ]:
df_flat['context_tokens'] = df_flat['context'].apply(lambda x: len(tokenizer(x)['input_ids']))
df_flat['question_tokens'] = df_flat['question'].apply(lambda x: len(tokenizer(x)['input_ids']))
df_flat['answer_tokens'] = df_flat['answer'].apply(lambda x: len(tokenizer(x)['input_ids']))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Context tokens distribution
axes[0, 0].hist(df_flat['context_tokens'], bins=50, edgecolor='black')
axes[0, 0].set_title('Context Token Length Distribution')
axes[0, 0].set_xlabel('Number of Tokens')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(np.mean(df_flat['context_tokens']), color='r', linestyle='--', label=f'Mean: {np.mean(df_flat["context_tokens"]):.1f}')
axes[0, 0].axvline(np.percentile(df_flat['context_tokens'], 50), color='g', linestyle='--', label=f'Median: {np.percentile(df_flat["context_tokens"], 50):.1f}')
axes[0, 0].legend()

# Question tokens distribution
axes[0, 1].hist(df_flat['question_tokens'], bins=50, edgecolor='black')
axes[0, 1].set_title('Question Token Length Distribution')
axes[0, 1].set_xlabel('Number of Tokens')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.mean(df_flat['question_tokens']), color='r', linestyle='--', label=f'Mean: {np.mean(df_flat["question_tokens"]):.1f}')
axes[0, 1].axvline(np.percentile(df_flat['question_tokens'], 50), color='g', linestyle='--', label=f'Median: {np.percentile(df_flat["question_tokens"], 50):.1f}')
axes[0, 1].legend()

# Answer tokens distribution
axes[1, 0].hist(df_flat['answer_tokens'], bins=50, edgecolor='black')
axes[1, 0].set_title('Answer Token Length Distribution')
axes[1, 0].set_xlabel('Number of Tokens')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(np.mean(df_flat['answer_tokens']), color='r', linestyle='--', label=f'Mean: {np.mean(df_flat["answer_tokens"]):.1f}')
axes[1, 0].axvline(np.percentile(df_flat['answer_tokens'], 50), color='g', linestyle='--', label=f'Median: {np.percentile(df_flat["answer_tokens"], 50):.1f}')
axes[1, 0].legend()

# Summary statistics table
axes[1, 1].axis('tight')
axes[1, 1].axis('off')
stats_data = [
    ['Metric', 'Context', 'Question', 'Answer'],
    ['Mean', f'{np.mean(df_flat["context_tokens"]):.1f}', f'{np.mean(df_flat["question_tokens"]):.1f}', f'{np.mean(df_flat["answer_tokens"]):.1f}'],
    ['Median', f'{np.median(df_flat["context_tokens"]):.1f}', f'{np.median(df_flat["question_tokens"]):.1f}', f'{np.median(df_flat["answer_tokens"]):.1f}'],
    ['Min', f'{np.min(df_flat["context_tokens"])}', f'{np.min(df_flat["question_tokens"])}', f'{np.min(df_flat["answer_tokens"])}'],
    ['Max', f'{np.max(df_flat["context_tokens"])}', f'{np.max(df_flat["question_tokens"])}', f'{np.max(df_flat["answer_tokens"])}'],
    ['Std Dev', f'{np.std(df_flat["context_tokens"]):.1f}', f'{np.std(df_flat["question_tokens"]):.1f}', f'{np.std(df_flat["answer_tokens"]):.1f}'],
    ['15%', f'{np.percentile(df_flat["context_tokens"], 15):.1f}', f'{np.percentile(df_flat["question_tokens"], 15):.1f}', f'{np.percentile(df_flat["answer_tokens"], 15):.1f}'],
    ['25%', f'{np.percentile(df_flat["context_tokens"], 25):.1f}', f'{np.percentile(df_flat["question_tokens"], 25):.1f}', f'{np.percentile(df_flat["answer_tokens"], 25):.1f}'],
    ['50%', f'{np.percentile(df_flat["context_tokens"], 50):.1f}', f'{np.percentile(df_flat["question_tokens"], 50):.1f}', f'{np.percentile(df_flat["answer_tokens"], 50):.1f}'],
    ['75%', f'{np.percentile(df_flat["context_tokens"], 75):.1f}', f'{np.percentile(df_flat["question_tokens"], 75):.1f}', f'{np.percentile(df_flat["answer_tokens"], 75):.1f}'],
    ['95%', f'{np.percentile(df_flat["context_tokens"], 95):.1f}', f'{np.percentile(df_flat["question_tokens"], 95):.1f}', f'{np.percentile(df_flat["answer_tokens"], 95):.1f}']
]
table = axes[1, 1].table(cellText=stats_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)

plt.tight_layout()
plt.show()

In [ ]:
# Inspect outliers for context tokens
print("=" * 60)
print("CONTEXT TOKEN OUTLIERS")
print("=" * 60)

# Find min and max context token indices
min_context_idx = np.argmin(df_flat['context_tokens'])
max_context_idx = np.argmax(df_flat['context_tokens'])

print(f"\n[Minimum Context Tokens: {df_flat['context_tokens'].iloc[min_context_idx]}]")
print(f"Context text:\n{df_flat['context'].iloc[min_context_idx]}\n")

print(f"[Maximum Context Tokens: {df_flat['context_tokens'].iloc[max_context_idx]}]")
print(f"Context text:\n{df_flat['context'].iloc[max_context_idx][:500]}...\n")  # Show first 500 chars

# Inspect outliers for questions
print("=" * 60)
print("QUESTION TOKEN OUTLIERS")
print("=" * 60)

min_q_idx = df_flat['question_tokens'].idxmin()
max_q_idx = df_flat['question_tokens'].idxmax()

print(f"\n[Minimum Question Tokens: {df_flat.loc[min_q_idx, 'question_tokens']}]")
print(f"Question: {df_flat.loc[min_q_idx, 'question']}\n")

print(f"[Maximum Question Tokens: {df_flat.loc[max_q_idx, 'question_tokens']}]")
print(f"Question: {df_flat.loc[max_q_idx, 'question']}\n")

# Inspect outliers for answers
print("=" * 60)
print("ANSWER TOKEN OUTLIERS")
print("=" * 60)

min_a_idx = df_flat['answer_tokens'].idxmin()
max_a_idx = df_flat['answer_tokens'].idxmax()

print(f"\n[Minimum Answer Tokens: {df_flat.loc[min_a_idx, 'answer_tokens']}]")
print(f"Answer: {df_flat.loc[min_a_idx, 'answer']}\n")

print(f"[Maximum Answer Tokens: {df_flat.loc[max_a_idx, 'answer_tokens']}]")
print(f"Answer: {df_flat.loc[max_a_idx, 'answer']}\n")

print("=" * 60)

# 3. Topic Distribution

In [ ]:
topic_ids = df_flat['topic_id'].value_counts().index.tolist()
topic_counts = df_flat['topic_id'].value_counts().values.tolist()

# Plot pie chart
plt.figure(figsize=(6, 6))
plt.pie(topic_counts, labels=topic_ids, autopct='%1.1f%%', startangle=140)
plt.title('Topic Distribution after Trimming')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

## 3b. Dataset Structure: Articles, Chunks & QA Pairs

Understanding how QA pairs are distributed across articles and text chunks is important for reporting dataset construction methodology and detecting imbalances.

In [ ]:
# Distribution of chunks per article and QA pairs per chunk
# (computed from df before flattening, so run after filtering cells)
chunks_per_article = [len(row.get('chunks', [])) for _, row in df.iterrows()]
qa_per_chunk = [
    len(chunk.get('qa_pairs', []))
    for _, row in df.iterrows()
    for chunk in row.get('chunks', [])
    if isinstance(chunk, dict)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(chunks_per_article, bins=range(1, max(chunks_per_article) + 2),
             edgecolor='black', color='steelblue', alpha=0.8, align='left')
axes[0].axvline(np.mean(chunks_per_article), color='r', linestyle='--',
                label=f'Mean: {np.mean(chunks_per_article):.1f}')
axes[0].axvline(np.median(chunks_per_article), color='g', linestyle='--',
                label=f'Median: {np.median(chunks_per_article):.0f}')
axes[0].set_title('Chunks per Article', fontsize=12)
axes[0].set_xlabel('Number of Chunks')
axes[0].set_ylabel('Number of Articles')
axes[0].legend()

axes[1].hist(qa_per_chunk, bins=range(1, max(qa_per_chunk) + 2),
             edgecolor='black', color='darkorange', alpha=0.8, align='left')
axes[1].axvline(np.mean(qa_per_chunk), color='r', linestyle='--',
                label=f'Mean: {np.mean(qa_per_chunk):.1f}')
axes[1].axvline(np.median(qa_per_chunk), color='g', linestyle='--',
                label=f'Median: {np.median(qa_per_chunk):.0f}')
axes[1].set_title('QA Pairs per Chunk', fontsize=12)
axes[1].set_xlabel('Number of QA Pairs')
axes[1].set_ylabel('Number of Chunks')
axes[1].legend()

plt.suptitle('Dataset Structural Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"  Total articles   : {len(df):,}")
print(f"  Chunks/article   : mean={np.mean(chunks_per_article):.1f}, median={np.median(chunks_per_article):.0f}, max={max(chunks_per_article)}")
print(f"  QA pairs/chunk   : mean={np.mean(qa_per_chunk):.1f}, median={np.median(qa_per_chunk):.0f}, max={max(qa_per_chunk)}")

# 4. Evaluation Score Distribution

In [ ]:
score_cols = ['answerability_score', 'answer_accuracy_score', 'clarity_score', 'conciseness_score', 'usefulness_score']

# Convert score columns to numeric
for col in score_cols:
    df_flat[col] = pd.to_numeric(df_flat[col], errors='coerce')
df_flat['total_score'] = pd.to_numeric(df_flat['total_score'], errors='coerce')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    data = df_flat[col].dropna()
    axes[i].hist(data, bins=20, edgecolor='black', color='steelblue', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12)
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(data.mean(), color='r', linestyle='--', label=f'Mean: {data.mean():.2f}')
    axes[i].axvline(data.median(), color='g', linestyle='--', label=f'Median: {data.median():.2f}')
    axes[i].legend(fontsize=9)

# Total score distribution
data = df_flat['total_score'].dropna()
axes[5].hist(data, bins=30, edgecolor='black', color='darkorange', alpha=0.8)
axes[5].set_title('Total Score', fontsize=12)
axes[5].set_xlabel('Score')
axes[5].set_ylabel('Frequency')
axes[5].axvline(data.mean(), color='r', linestyle='--', label=f'Mean: {data.mean():.2f}')
axes[5].axvline(data.median(), color='g', linestyle='--', label=f'Median: {data.median():.2f}')
axes[5].legend(fontsize=9)

plt.suptitle('Evaluation Score Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics table
print("=" * 80)
print("SCORE SUMMARY STATISTICS")
print("=" * 80)
score_stats = df_flat[score_cols + ['total_score']].describe().round(2)
print(score_stats.to_string())
print("=" * 80)

# 5. Quality Assessment Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors_map = {'ACCEPT': '#2ecc71', 'REVISE': '#f39c12', 'REJECT': '#e74c3c'}

# Overall assessment distribution
assessment_counts = df_flat['overall_assessment'].value_counts()
colors = [colors_map.get(label, '#3498db') for label in assessment_counts.index]
axes[0].bar(assessment_counts.index, assessment_counts.values, color=colors, edgecolor='black')
for i, (label, count) in enumerate(zip(assessment_counts.index, assessment_counts.values)):
    axes[0].text(i, count + len(df_flat) * 0.005,
                 f'{count}\n({count/len(df_flat):.1%})', ha='center', fontsize=10)
axes[0].set_title('Overall Assessment Distribution', fontsize=12)
axes[0].set_ylabel('Count')

# Accept rate by topic — use boolean mean to avoid FutureWarning from groupby.apply
accept_by_topic = (
    (df_flat['overall_assessment'] == 'ACCEPT')
    .groupby(df_flat['topic_id'])
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)
bars = axes[1].bar(accept_by_topic.index, accept_by_topic.values, color='steelblue', edgecolor='black')
for bar, val in zip(bars, accept_by_topic.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=10)
axes[1].set_title('Accept Rate by Topic', fontsize=12)
axes[1].set_ylabel('Accept Rate (%)')
axes[1].set_ylim(0, 105)

# Total score box plot grouped by overall assessment
assessment_groups = [g for g in ['ACCEPT', 'REVISE', 'REJECT'] if g in df_flat['overall_assessment'].values]
data_by_group = [df_flat[df_flat['overall_assessment'] == g]['total_score'].dropna().values
                 for g in assessment_groups]
valid = [(g, d) for g, d in zip(assessment_groups, data_by_group) if len(d) > 0]
if valid:
    groups, data = zip(*valid)
    bp = axes[2].boxplot(data, labels=groups, patch_artist=True)
    for patch, group in zip(bp['boxes'], groups):
        patch.set_facecolor(colors_map.get(group, '#3498db'))
        patch.set_alpha(0.6)
axes[2].set_title('Total Score by Assessment', fontsize=12)
axes[2].set_xlabel('Overall Assessment')
axes[2].set_ylabel('Total Score')

plt.tight_layout()
plt.show()

## 5b. Assessment Quality by Topic: Cross-Tabulation

Per-topic quality breakdown — reveals whether certain question types (FACTOID, SUMMARY, etc.) are systematically harder to synthesize well-formed QA pairs for.

In [ ]:
# Cross-tabulation: topic × overall_assessment
# Identifies whether any topic type is systematically harder to generate high-quality QA for.

cross_tab_abs = pd.crosstab(df_flat['topic_id'], df_flat['overall_assessment'])
cross_tab_pct = cross_tab_abs.div(cross_tab_abs.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap (percentage)
sns.heatmap(cross_tab_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[0], linewidths=0.5, cbar_kws={'label': 'Percentage (%)'},
            vmin=0, vmax=100)
axes[0].set_title('Assessment Rate by Topic (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Overall Assessment')
axes[0].set_ylabel('Topic')

# Stacked bar (absolute counts)
colors_assessment = {'ACCEPT': '#2ecc71', 'REVISE': '#f39c12', 'REJECT': '#e74c3c'}
bottom = pd.Series(0, index=cross_tab_abs.index)
for col in cross_tab_abs.columns:
    color = colors_assessment.get(col, '#95a5a6')
    axes[1].bar(cross_tab_abs.index, cross_tab_abs[col], bottom=bottom,
                label=col, color=color, edgecolor='black', alpha=0.85)
    bottom = bottom + cross_tab_abs[col]
axes[1].set_title('Assessment Count by Topic', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Topic')
axes[1].set_ylabel('Count')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

print("\nCross-tabulation (raw counts):")
print(pd.crosstab(df_flat['topic_id'], df_flat['overall_assessment'], margins=True).to_string())

# 6. Score Correlation Analysis

In [ ]:
# sns is already imported at the top — no re-import needed
corr_cols = score_cols + ['total_score', 'context_tokens', 'question_tokens', 'answer_tokens']
corr_matrix = df_flat[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in corr_cols],
            yticklabels=[c.replace('_', '\n') for c in corr_cols])
ax.set_title('Correlation Heatmap: Scores & Token Lengths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 7. Topic-wise Score Analysis

In [ ]:
# Grouped bar chart: mean score per dimension by topic
topic_score_means = df_flat.groupby('topic_id')[score_cols].mean()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(topic_score_means.index))
width = 0.15
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, col in enumerate(score_cols):
    bars = ax.bar(x + i * width, topic_score_means[col], width, label=col.replace('_score', '').title(), color=colors[i], edgecolor='black', alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xlabel('Topic ID')
ax.set_ylabel('Mean Score')
ax.set_title('Mean Scores by Topic and Dimension', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(topic_score_means.index)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
plt.tight_layout()
plt.show()

# Table view
print("=" * 80)
print("MEAN SCORES BY TOPIC")
print("=" * 80)
topic_summary = df_flat.groupby('topic_id').agg(
    count=('total_score', 'size'),
    total_score_mean=('total_score', 'mean'),
    total_score_std=('total_score', 'std'),
    **{f'{col}_mean': (col, 'mean') for col in score_cols}
).round(2)
print(topic_summary.to_string())
print("=" * 80)

In [ ]:
# Box plot: total score distribution by topic
# notch=True removed — produces UserWarning when the notch extends beyond whiskers for small groups
fig, ax = plt.subplots(figsize=(10, 6))
topics = df_flat['topic_id'].unique()
data_by_topic = [df_flat[df_flat['topic_id'] == t]['total_score'].dropna().values for t in topics]

bp = ax.boxplot(data_by_topic, labels=topics, patch_artist=True)
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
for patch, color in zip(bp['boxes'], colors[:len(topics)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_title('Total Score Distribution by Topic', fontsize=13, fontweight='bold')
ax.set_xlabel('Topic')
ax.set_ylabel('Total Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Token Length vs Score Relationship

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

token_cols = ['context_tokens', 'question_tokens', 'answer_tokens']
titles = ['Context Length vs Total Score', 'Question Length vs Total Score', 'Answer Length vs Total Score']

for i, (tcol, title) in enumerate(zip(token_cols, titles)):
    valid = df_flat[[tcol, 'total_score']].dropna()
    axes[i].scatter(valid[tcol], valid['total_score'], alpha=0.2, s=10, color='steelblue')
    # Trend line
    z = np.polyfit(valid[tcol], valid['total_score'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(valid[tcol].min(), valid[tcol].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.4f})')
    axes[i].set_title(title, fontsize=12)
    axes[i].set_xlabel('Token Count')
    axes[i].set_ylabel('Total Score')
    axes[i].legend(fontsize=9)
    axes[i].grid(alpha=0.3)

plt.suptitle('Token Length vs Total Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 8b. Answer Extractiveness Analysis

Measures how much answer text is sourced verbatim from the context vs. paraphrased/inferred. This characterises dataset difficulty and is a key descriptor for ML conference submissions — models must retrieve rather than memorise for low-overlap pairs.

In [ ]:
# Measure how extractive answers are relative to their source context.
#   Jaccard similarity = |tokens(A) ∩ tokens(C)| / |tokens(A) ∪ tokens(C)|
#   High score → extractive; Low score → abstractive / inferential.
# Also checks for verbatim (substring) matches.

def token_jaccard(text_a: str, text_b: str) -> float:
    tokens_a = set(text_a.lower().split())
    tokens_b = set(text_b.lower().split())
    if not tokens_a or not tokens_b:
        return 0.0
    return len(tokens_a & tokens_b) / len(tokens_a | tokens_b)

def is_verbatim(answer: str, context: str) -> bool:
    return answer.strip().lower() in context.lower()

df_flat['ans_ctx_jaccard'] = df_flat.apply(
    lambda r: token_jaccard(r['answer'], r['context']), axis=1)
df_flat['is_extractive'] = df_flat.apply(
    lambda r: is_verbatim(r['answer'], r['context']), axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of Jaccard scores
axes[0].hist(df_flat['ans_ctx_jaccard'], bins=40, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].axvline(df_flat['ans_ctx_jaccard'].mean(), color='r', linestyle='--',
                label=f"Mean: {df_flat['ans_ctx_jaccard'].mean():.3f}")
axes[0].axvline(df_flat['ans_ctx_jaccard'].median(), color='g', linestyle='--',
                label=f"Median: {df_flat['ans_ctx_jaccard'].median():.3f}")
axes[0].set_title('Answer-Context Token Overlap (Jaccard)', fontsize=12)
axes[0].set_xlabel('Jaccard Similarity')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Mean Jaccard by topic
topic_jaccard = df_flat.groupby('topic_id')['ans_ctx_jaccard'].mean().sort_values(ascending=False)
bars = axes[1].bar(topic_jaccard.index, topic_jaccard.values, color='darkorange', edgecolor='black', alpha=0.85)
for bar, val in zip(bars, topic_jaccard.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', fontsize=10)
axes[1].set_title('Mean Jaccard by Topic', fontsize=12)
axes[1].set_xlabel('Topic')
axes[1].set_ylabel('Mean Jaccard Similarity')
axes[1].set_ylim(0, topic_jaccard.max() * 1.18)

# Verbatim vs paraphrased
ext_count = df_flat['is_extractive'].sum()
abst_count = len(df_flat) - ext_count
labels = ['Verbatim\n(extractive)', 'Paraphrased\n(abstractive)']
axes[2].bar(labels, [ext_count, abst_count],
            color=['#2ecc71', '#e74c3c'], edgecolor='black', alpha=0.85)
for i, v in enumerate([ext_count, abst_count]):
    axes[2].text(i, v + len(df_flat)*0.005,
                 f'{v:,}\n({v/len(df_flat):.1%})', ha='center', fontsize=10)
axes[2].set_title('Verbatim Answer in Context', fontsize=12)
axes[2].set_ylabel('Count')

plt.suptitle('Answer Extractiveness Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean Jaccard similarity : {df_flat['ans_ctx_jaccard'].mean():.4f}")
print(f"Verbatim answers        : {ext_count:,} ({df_flat['is_extractive'].mean():.1%})")
print(f"Paraphrased answers     : {abst_count:,} ({1 - df_flat['is_extractive'].mean():.1%})")

# 9. Data Quality & Duplicate Analysis

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# Missing values
print("\n--- Missing Values ---")
missing = df_flat[['context', 'question', 'answer', 'topic_id', 'total_score', 'overall_assessment']].isnull().sum()
for col, count in missing.items():
    print(f"  {col}: {count} ({count/len(df_flat):.2%})")

# Duplicate questions
dup_questions = df_flat['question'].duplicated().sum()
print(f"\n--- Duplicates ---")
print(f"  Duplicate questions: {dup_questions} ({dup_questions/len(df_flat):.2%})")

# Duplicate answers
dup_answers = df_flat['answer'].duplicated().sum()
print(f"  Duplicate answers: {dup_answers} ({dup_answers/len(df_flat):.2%})")

# Duplicate (question, answer) pairs
dup_qa = df_flat.duplicated(subset=['question', 'answer']).sum()
print(f"  Duplicate QA pairs: {dup_qa} ({dup_qa/len(df_flat):.2%})")

# Unique contexts
unique_contexts = df_flat['context'].nunique()
print(f"\n--- Unique Counts ---")
print(f"  Unique contexts: {unique_contexts}")
print(f"  Unique questions: {df_flat['question'].nunique()}")
print(f"  Unique answers: {df_flat['answer'].nunique()}")
print(f"  QA pairs per unique context: {len(df_flat)/unique_contexts:.2f}")

# Empty or very short text checks
print(f"\n--- Short Text Check ---")
short_q = (df_flat['question'].str.len() < 10).sum()
short_a = (df_flat['answer'].str.len() < 5).sum()
print(f"  Questions < 10 chars: {short_q}")
print(f"  Answers < 5 chars: {short_a}")

print("=" * 60)

## 9b. Temporal Distribution of Articles

Visualises when source articles were published. Temporal coverage and potential skew should be reported in any dataset paper to allow readers to assess out-of-distribution risks.

In [ ]:
# Temporal coverage of the article corpus.
# Shows how articles are distributed over time, making dataset temporal bias visible —
# a standard dataset characterisation expected in academic submissions.

df_time = df_flat.copy()
df_time['time_parsed'] = pd.to_datetime(df_time['time'], errors='coerce')
df_time['year_month'] = df_time['time_parsed'].dt.to_period('M')

# Deduplicate to article level for the article-count chart
df_articles_time = df_time.drop_duplicates(subset=['url'])

# dropna() prevents NaT periods from appearing as a "NaT" bar in the charts
monthly_articles = df_articles_time['year_month'].dropna().value_counts().sort_index()
monthly_qa = df_time.dropna(subset=['year_month']).groupby('year_month').size().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(monthly_articles.index.astype(str), monthly_articles.values,
            edgecolor='black', color='steelblue', alpha=0.8)
axes[0].set_title('Articles per Month', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Year-Month')
axes[0].set_ylabel('Number of Articles')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

axes[1].bar(monthly_qa.index.astype(str), monthly_qa.values,
            edgecolor='black', color='darkorange', alpha=0.8)
axes[1].set_title('QA Pairs per Month', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Year-Month')
axes[1].set_ylabel('Number of QA Pairs')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

plt.suptitle('Temporal Distribution of the Corpus', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

valid_dates = df_articles_time['time_parsed'].dropna()
if len(valid_dates):
    print(f"Date range      : {valid_dates.min().date()} → {valid_dates.max().date()}")
    print(f"Span            : {(valid_dates.max() - valid_dates.min()).days} days")
    print(f"Months covered  : {monthly_articles.shape[0]}")
    print(f"Peak month      : {monthly_articles.idxmax()} ({monthly_articles.max()} articles)")

## 9c. Question Type Distribution (WH-Word Analysis)

Linguistic characterisation of question types using Vietnamese interrogative words. Demonstrates vocabulary variety and coverage of question categories — critical evidence for linguistics conference papers.

In [ ]:
# Vietnamese WH-word based question type analysis.
# Captures the leading interrogative word/phrase of each question to characterise
# the linguistic variety of the dataset — essential for a linguistics conference submission.

WH_PATTERNS = [
    (r'^(ai\b|ai là|ai đã|ai sẽ|ai đang|ai được|ai có thể)', 'Who (Ai)'),
    (r'^(điều gì|những gì|cái gì|thứ gì|gì\b)', 'What (Gì)'),
    (r'^(khi nào|lúc nào|bao giờ|thời điểm nào|thời gian nào)', 'When (Khi nào)'),
    (r'^(ở đâu|tại đâu|nơi nào|địa điểm nào)', 'Where (Ở đâu)'),
    (r'^(tại sao|vì sao|lý do gì|nguyên nhân gì)', 'Why (Tại sao)'),
    (r'^(như thế nào|ra sao|thế nào|bằng cách nào|làm thế nào|làm sao)', 'How (Như thế nào)'),
    (r'^(bao nhiêu|bao lâu|bao xa|mấy\b)', 'How much/many (Bao nhiêu)'),
]

def classify_question(q: str) -> str:
    # Strip the generation prefix (e.g. "FACTOID-", "SUMMARY-") BEFORE applying the
    # ^ anchored patterns. Without this, the Vietnamese WH words are never at position 0
    # and every question would fall through to "Other".
    q_stripped = re.sub(
        r'^(FACTOID|SUMMARY|COMPARISON|VERIFICATION)-', '', q, flags=re.IGNORECASE
    ).strip()
    q_lower = q_stripped.lower()
    for pattern, label in WH_PATTERNS:
        if re.search(pattern, q_lower):
            return label
    return 'Other'

df_flat['question_type'] = df_flat['question'].apply(classify_question)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Horizontal bar chart — overall distribution
qt_counts = df_flat['question_type'].value_counts()
axes[0].barh(qt_counts.index[::-1], qt_counts.values[::-1],
             color='steelblue', edgecolor='black', alpha=0.85)
for i, (label, val) in enumerate(zip(qt_counts.index[::-1], qt_counts.values[::-1])):
    axes[0].text(val + len(df_flat) * 0.003, i,
                 f'{val:,} ({val/len(df_flat):.1%})', va='center', fontsize=9)
axes[0].set_title('Question Type Distribution (WH-word)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].set_xlim(0, qt_counts.max() * 1.22)

# Stacked percentage bar by topic
qt_topic = df_flat.groupby(['topic_id', 'question_type']).size().unstack(fill_value=0)
qt_topic_pct = qt_topic.div(qt_topic.sum(axis=1), axis=0) * 100
qt_topic_pct.plot(kind='bar', ax=axes[1], edgecolor='black', alpha=0.85)
axes[1].set_title('Question Type by Topic (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Topic')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.45, 1.0))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\nQuestion Type Summary:")
for qtype, count in qt_counts.items():
    print(f"  {qtype:<35} {count:>6,}  ({count/len(df_flat):.1%})")

# 10. Low-score Sample Inspection

In [ ]:
# Inspect worst-scoring QA pairs
n_samples = 5
worst = df_flat.nsmallest(n_samples, 'total_score')[['topic_id', 'question', 'answer', 'total_score', 'overall_assessment', 'overall_reason']]

print(f"===== Bottom {n_samples} QA Pairs by Total Score =====\n")
for i, (_, row) in enumerate(worst.iterrows(), 1):
    print(f"--- Sample {i} (score={row['total_score']}, assessment={row['overall_assessment']}) ---")
    print(f"  Topic: {row['topic_id']}")
    print(f"  Q: {row['question'][:200]}")
    print(f"  A: {row['answer'][:200]}")
    print(f"  Reason: {row['overall_reason'][:300]}")
    print()

# Inspect best-scoring QA pairs
best = df_flat.nlargest(n_samples, 'total_score')[['topic_id', 'question', 'answer', 'total_score', 'overall_assessment', 'overall_reason']]

print(f"===== Top {n_samples} QA Pairs by Total Score =====\n")
for i, (_, row) in enumerate(best.iterrows(), 1):
    print(f"--- Sample {i} (score={row['total_score']}, assessment={row['overall_assessment']}) ---")
    print(f"  Topic: {row['topic_id']}")
    print(f"  Q: {row['question'][:200]}")
    print(f"  A: {row['answer'][:200]}")
    print(f"  Reason: {row['overall_reason'][:300]}")
    print()

# 11. Final Dataset Summary

In [ ]:
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

accept_count = (df_flat['overall_assessment'] == 'ACCEPT').sum()
revise_count = (df_flat['overall_assessment'] == 'REVISE').sum()
reject_count = (df_flat['overall_assessment'] == 'REJECT').sum()

print(f"\n[Scale]")
print(f"  Total QA pairs (after trimming): {len(df_flat):,}")
print(f"  Unique contexts                : {df_flat['context'].nunique():,}")
print(f"  Unique articles                : {df_flat['url'].nunique():,}")
print(f"  Topics                         : {', '.join(sorted(df_flat['topic_id'].unique()))}")

print(f"\n[Quality Assessment]")
print(f"  ACCEPT : {accept_count:,} ({accept_count/len(df_flat):.1%})")
print(f"  REVISE : {revise_count:,} ({revise_count/len(df_flat):.1%})")
print(f"  REJECT : {reject_count:,} ({reject_count/len(df_flat):.1%})")

print(f"\n[Evaluation Scores]")
print(f"  Mean total score : {df_flat['total_score'].mean():.2f} ± {df_flat['total_score'].std():.2f}  (max 10)")
for col in ['answerability', 'answer_accuracy', 'clarity', 'conciseness', 'usefulness']:
    avg = df_flat[f'{col}_score'].dropna().mean()
    print(f"  {col:<20}: {avg:.2f} / 2")

print(f"\n[Token Length Ranges]")
print(f"  Context  : {df_flat['context_tokens'].min()} – {df_flat['context_tokens'].max()} (mean: {df_flat['context_tokens'].mean():.0f})")
print(f"  Question : {df_flat['question_tokens'].min()} – {df_flat['question_tokens'].max()} (mean: {df_flat['question_tokens'].mean():.0f})")
print(f"  Answer   : {df_flat['answer_tokens'].min()} – {df_flat['answer_tokens'].max()} (mean: {df_flat['answer_tokens'].mean():.0f})")

if 'ans_ctx_jaccard' in df_flat.columns:
    print(f"\n[Answer Extractiveness]")
    print(f"  Mean Jaccard similarity : {df_flat['ans_ctx_jaccard'].mean():.4f}")
    print(f"  Verbatim answers        : {df_flat['is_extractive'].mean():.1%}")
    print(f"  Paraphrased answers     : {(~df_flat['is_extractive']).mean():.1%}")

if 'question_type' in df_flat.columns:
    dominant_type = df_flat['question_type'].mode()[0]
    dominant_pct = (df_flat['question_type'] == dominant_type).mean()
    print(f"\n[Question Linguistics]")
    print(f"  Dominant type : {dominant_type} ({dominant_pct:.1%})")
    for qtype, cnt in df_flat['question_type'].value_counts().items():
        print(f"  {qtype:<35} {cnt:>6,}  ({cnt/len(df_flat):.1%})")

print("=" * 60)

In [ ]:
final_cols = [
    "url",
    "title",
    "time",
    "context",
    "question",
    "answer",
    "question_type",
]

import re

def remove_topic_prefix(q: str) -> str:
    return re.sub(r'^(FACTOID|SUMMARY|COMPARISON|VERIFICATION)-', '', q, flags=re.IGNORECASE).strip()


df_flat['question_type'] = df_flat['topic_id']
df_flat['question'] = df_flat['question'].apply(remove_topic_prefix)
df_flat = df_flat[df_flat['overall_assessment'] == "ACCEPT"]
df_flat = df_flat[final_cols]
df_flat.head()

In [ ]:
# df_flat.to_csv("../data/processed/QA_data_cleaned.csv", index=False)

In [ ]:
df_flat = df_flat.reset_index(drop=True)
df_flat.info()

In [ ]:
# unique articles
print(f"Unique articles: {df_flat['url'].nunique():,}")